## LangGraph-Orchestrated LLM-Based Optimization Agent

This implements the first **LLM-based autonomous optimization agent**. 

**LangGraph** orchestrates the optimization workflow, while the reflection node (in *agent.py* file) uses **GPT-5** to reason over previous experiments and the current clustering evaluation metrics before suggesting the next set of clustering parameters.

The objective of this phase is not to find the optimal clustering parameters, but to validate that an LLM can act as the reasoning component within an iterative optimization framework.

Specifically, this phase verifies that the agent can:

* Run the clustering and evaluation tools automatically through LangGraph.
* Observe and evaluate clustering quality using the internal evaluation metrics.
* Remember previous optimization experiments through the graph state.
* Reason over the optimization history using an LLM.
* Return validated parameter suggestions using a **Pydantic** structured output model.
* Decide whether the optimization should continue or stop.

### Implementing Guardrails
To ensure the optimization remains efficient and reproducible, the workflow introduces its first deterministic guardrail
+ **a maximum iteration limit** 

Rather than allowing the LLM to continue proposing new experiments indefinitely, the framework enforces a predefined optimization budget. The guardrail is evaluated **before** the LLM is invoked, preventing unnecessary API calls and reducing execution time from approximately **686 seconds** to **90 seconds** during testing.

Overall, at this stage:
+ The architecture combines deterministic execution with adaptive LLM reasoning. 
+ LangGraph manages the workflow
+ Deterministic guardrails govern the optimization process
+ The LLM is responsible for interpreting the evaluation metrics and proposing the next clustering parameters. 

Overtime, additional guardrails, experiment logging, stability analysis, and automated reporting will be introduced in the subsequent phases.


## Modules Used in This Phase

| Module                                   | Purpose                                                                                                                                                                                     |
| ---------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **state.py**                             | Defines the LangGraph state, including the dataset (`AnnData`), clustering parameters, evaluation metrics, optimization history, iteration counter, decision, and reasoning.                |
| **clustering.py**                        | Implements the deterministic clustering tool. Runs the neighborhood graph construction and Leiden clustering using the current optimization parameters.                                     |
| **evaluation.py**                        | Computes the clustering evaluation metrics (e.g., Silhouette Score and Davies–Bouldin Index) and records each optimization experiment in the graph state.                                   |
| **agent.py**                             | Defines the graph nodes (`cluster_node`, `evaluate_node`, and `reflection_node`). The reflection node performs LLM-based reasoning, applies guardrails, and updates the optimization state. |
| **graph.py**                             | Builds the LangGraph workflow, connects the nodes, defines the conditional optimization loop, and compiles the executable graph.                                                            |
| **decision.py**                          | Defines the Pydantic models (`SuggestedParameters` and `OptimizationDecision`) used to validate the structured output returned by the LLM.                                                  |
| **prompts.py**                           | Contains the prompt template used by the reflection node. It formats the optimization history, parameters, and evaluation metrics into a prompt for the LLM.                                |
| **openAI_llm.py** *(or `llm.py`)*        | Configures the GPT-5 model used for structured reasoning during the reflection step.                                                                                                        |
| **optimization.ipynb** *(this notebook)* | Initializes the graph state, executes the LangGraph workflow, validates the optimization results, and demonstrates the complete LLM-driven optimization loop.                               |


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from dotenv import load_dotenv, find_dotenv
import os

dotenv_path = find_dotenv()
print("Found:", dotenv_path)

load_dotenv("/Users/lola/Desktop/scanpy_project/.env", override=True) # override=True forces a clean injection into os.environ

#print("API key:", os.getenv("OPENAI_API_KEY"))

Found: 


True

In [3]:
from src import data_load as dl
from src import preprocessing_pipeline as prp
from src import clustering as cl
from src import evaluation as ev
from src import optimization as opt
from src import visualisation as viz
from src import marker as mk
from src.graph import graph
from src.agent import cluster_node, evaluate_node, manual_reflection_node, reflection_node
from src.decision import OptimizationDecision
from src.prompts import build_reflection_prompt

import scanpy as sc
import pandas as pd
from langchain_openai.chat_models import ChatOpenAI 

### Create the LLM (OPENAI)

In [4]:
llm = ChatOpenAI(model="gpt-5", temperature=0,)

In [5]:
structured_llm = llm.with_structured_output(OptimizationDecision) #This help to return an optimized decision

## Fixed Preprocessing Pipeline

In [6]:
pbmc_data = dl.load_pbmc3k()
pbmc_data.var_names_make_unique()
pbmc_data = prp.run_preprocessing_pipeline(pbmc_data)

Running QC...
Running normalization...
Running PCA...
Finished preprocessing.


In [7]:
pbmc_data.var["highly_variable"].sum()

np.int64(2000)

## Agent State Setup

In [8]:
state = {
    "messages": [],
    "adata": pbmc_data,

    "parameters": {
        "n_neighbors": 15,
        "resolution": 1.0,
        "n_pcs": 20,
    },

    "metrics": {},
    "history": [],
    "iteration": 0,
    "decision": "continue",
    "reason": "",
}

In [9]:
result = graph.invoke(state)

In [10]:
history = result["history"]

parameters = result["parameters"]

metrics = result["metrics"]

In [11]:
state.keys()

dict_keys(['messages', 'adata', 'parameters', 'metrics', 'history', 'iteration', 'decision', 'reason'])

In [13]:
prompt = f"""
You are optimizing Leiden clustering.
Your objective is to maximize the Silhouette Score.
Minimize the Davies–Bouldin Index.
Adjust only one parameter per iteration to make the optimization interpretable.
Use the optimization history to avoid repeating unsuccessful parameter combinations.
Return the answer using the provided schema.

Current parameters:
{parameters}

Current metrics:
{metrics}

Optimization history:
{history}

Should optimization continue?

If yes, suggest updated clustering parameters.

Return your answer using the provided schema.
"""

In [14]:
decision = structured_llm.invoke(prompt)

In [15]:
print(type(decision))

<class 'src.decision.OptimizationDecision'>


In [ ]:
# print(decision)

In [17]:
print(decision.parameters.n_neighbors)
print(decision.parameters.resolution)
print(decision.parameters.n_pcs)

print(decision.reason)

20
0.8
20
Current best is at n_neighbors=15, resolution=0.8, n_pcs=20. Adjusting n_pcs and resolution previously worsened metrics; now vary only n_neighbors to 20 to potentially improve cohesion/separation and avoid repeating prior combinations.


In [9]:
#print(result["history"][0]["parameters"])

In [20]:
for experiment in result["history"]:
    print(
        experiment["iteration"],
        experiment["parameters"],
        experiment["metrics"]["silhouette_score"],
        experiment["metrics"]["davies_bouldin_score"],
    )

0 {'n_neighbors': 15, 'resolution': 1.0, 'n_pcs': 20} 0.07478884607553482 2.666380841974742
1 {'n_neighbors': 15, 'resolution': 1.0, 'n_pcs': 30} 0.06962878257036209 2.7087095145901565
2 {'n_neighbors': 15, 'resolution': 1.0, 'n_pcs': 20} 0.07478884607553482 2.666380841974742
3 {'n_neighbors': 15, 'resolution': 0.8, 'n_pcs': 20} 0.07574118673801422 2.6534070092029904
4 {'n_neighbors': 15, 'resolution': 0.8, 'n_pcs': 15} 0.0736665204167366 2.762290417012731
5 {'n_neighbors': 15, 'resolution': 0.8, 'n_pcs': 20} 0.07574118673801422 2.6534070092029904


This shows the LLM is:

+ reading optimization history,
+ identifying the best experiment,
+ avoiding changing multiple parameters at once

**But the LLM occasionally proposed parameter combinations that had already been evaluated**

### Current State Info

In [19]:
print(result["iteration"])
print(result["decision"])

5
stop


The iteration being 5 shows that the agent stopped at the sixth iteraction as designed.